We're studying different World cups to analyze its Macroeconomic impact in the hosting Country.
For control we will use a country with similar attributes to the hosting country.

DATA TO GATHER:
MacroEconomic: GDP, GDP Growth, Tourism Revenue, Inflation Rate, Government Debt (% of GDP), Unemployment Rate
Tourism Data: International Tourist Arrivals
Investment: Public Infrastructure spending if a reliable source is found later

DATA PERIOD
We will gather 5 years prior and after the world cup to analyze which will depend for each host
Germany - 2006
South Africa -  2010
Brazil - 2014 
Russia - 2018


Main APIs used here:
1. World Bank API macro/tourism indicators 
2. IMF DataMapper API can be used later for government debt if World Bank has NaN values

Webscraping
Used Webscraping to get world cup logo for the presentation. 
Gathered from Wikipedia

SECTION 2: DATA PLAN AND INDICATOR CODES
World Bank indicators: https://data.worldbank.org/indicator?tab=all

Example with Germany:
Germany country code in World Bank API: DEU
GDP current US$ - NY.GDP.MKTP.CD
GDP growth annual (%) - NY.GDP.MKTP.KD.ZG
Inflation rate (%) - FP.CPI.TOTL.ZG
Unemployment rate (%) - SL.UEM.TOTL.ZS
International Tourism arrivals - ST.INT.ARVL
Tourism receipts current US$ - ST.INT.RCPT.CD
Tourism receipts as % of GDP - calculated as tourism receipts / GDP * 100

In [2]:
# SECTION 3: IMPORT LIBRARIES AND DISPLAY SETTINGS


# pandas and numpy support data manipulation and calculations
import pandas as pd
import numpy as np
# requests retrieves data from the World Bank and IMF APIs
import requests
# matplotlib supports later visual analysis - only ussed for certain solutions
import matplotlib.pyplot as plt

# Display large numbers with commas and two decimal places
pd.set_option("display.float_format", "{:,.2f}".format)


In [3]:
# SECTION 4: WORLD BANK API HELPER FUNCTION
# Retrieve one indicator from the World Bank API and prepare it for analysis
def get_worldbank_indicator(country, indicator, metric_name):

    # Build the API request using the country and indicator codes
    url_worldbank = f"https://api.worldbank.org/v2/country/{country}/indicator/{indicator}?format=json&per_page=100"

    # Request the data and stop if the API returns an error
    response = requests.get(url_worldbank)
    response.raise_for_status()

    # Convert the API response into a two-column DataFrame
    data = response.json()

    df_country = pd.DataFrame(data[1])

    df_country = df_country[["date", "value"]]

    df_country = df_country.rename(columns={
        "date": "year",
        "value": metric_name
    })

    # Convert year to an integer and keep the 2001-2011 study period
    df_country["year"] = df_country["year"].astype(int)

    df_country = df_country[df_country["year"].between(2001, 2011)]

    return df_country.sort_values("year").reset_index(drop=True)

In [4]:
# SECTION 5: IMF API HELPER FUNCTION
# Retrieve one indicator from the IMF DataMapper API
def get_IMF_indicator(country, indicator, metric_name):

    # Build and send the request for the selected IMF indicator
    url_IMF = f"https://www.imf.org/external/datamapper/api/v2/{indicator}"

    response = requests.get(url_IMF)
    response.raise_for_status()

    data = response.json()

    # Select the year-value dictionary for the requested country
    country_data = data["values"][indicator][country]

    df_IMF_country = pd.DataFrame(
        country_data.items(),
        columns=["year", metric_name]
    )

    # Convert year to an integer and keep the 2001-2011 study period
    df_IMF_country["year"] = df_IMF_country["year"].astype(int)

    df_IMF_country = df_IMF_country[
        df_IMF_country["year"].between(2001, 2011)
    ]

    return df_IMF_country.sort_values("year").reset_index(drop=True)

In [5]:
# SECTION 6: BUILD EACH COUNTRY'S DATASET
# Build one complete macroeconomic and tourism dataset for a country
def build_country_dataset(wb_code, imf_code, country_name):

    # Actual GDP measured in current US dollars
    gdp = get_worldbank_indicator(
        wb_code,
        "NY.GDP.MKTP.CD",
        "gdp_current_usd"
    )

    # Annual GDP growth rate expressed as a percentage
    gdp_growth = get_worldbank_indicator(
        wb_code,
        "NY.GDP.MKTP.KD.ZG",
        "gdp_growth_pct"
    )

    # Annual inflation rate expressed as a percentage
    inflation = get_worldbank_indicator(
        wb_code,
        "FP.CPI.TOTL.ZG",
        "inflation_pct"
    )

    # Unemployment as a percentage of the labour force
    unemployment = get_worldbank_indicator(
        wb_code,
        "SL.UEM.TOTL.ZS",
        "unemployment_pct"
    )

    # International tourism revenue measured in current US dollars
    tourism_receipts = get_worldbank_indicator(
        wb_code,
        "ST.INT.RCPT.CD",
        "tourism_revenue_current_usd"
    )

    # Government debt expressed as a percentage of GDP
    government_debt = get_IMF_indicator(
        imf_code,
        "GG_DEBT_GDP",
        "government_debt_pct_gdp"
    )

    # Merge all indicators by year; outer joins preserve incomplete years
    country_df = (
        gdp
        .merge(gdp_growth, on="year", how="outer")
        .merge(inflation, on="year", how="outer")
        .merge(unemployment, on="year", how="outer")
        .merge(tourism_receipts, on="year", how="outer")
        .merge(government_debt, on="year", how="outer")
    )

    # Calculate how much tourism revenue contributes relative to GDP
    country_df["tourism_revenue_pct_gdp"] = (
        country_df["tourism_revenue_current_usd"]
        / country_df["gdp_current_usd"]
        * 100
    )

    # Place the country name immediately after the year column
    country_df.insert(1, "country", country_name)

    return country_df

In [6]:
#USING GERMANY AS HOST AND FRANCE AS CONTROL


# SECTION 7: COMBINE GERMANY AND FRANCE
# Build the individual country datasets using their API country codes
germany_df = build_country_dataset("DEU", "DEU", "Germany")
france_df = build_country_dataset("FRA", "FRA", "France")

# Stack both countries into one analysis-ready table
germany_france_df = pd.concat(
    [germany_df, france_df],
    ignore_index=True
)

# Organize the rows by country and year
germany_france_df = germany_france_df.sort_values(
    ["country", "year"]
).reset_index(drop=True)

germany_france_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_revenue_pct_gdp
0,2001,France,"1,370,376,677,298.86",1.90,1.63,8.61,"37,826,000,000.00",59.32,2.76
1,2002,France,"1,492,427,756,382.43",1.07,1.92,8.70,"41,113,000,000.00",61.26,2.75
2,2003,France,"1,835,095,983,049.09",0.97,2.10,8.31,"46,576,000,000.00",65.36,2.54
3,2004,France,"2,109,792,297,237.03",2.87,2.14,8.91,"53,068,000,000.00",66.88,2.52
4,2005,France,"2,192,146,403,028.17",1.89,1.75,8.88,"52,126,000,000.00",68.19,2.38
5,2006,France,"2,317,861,544,690.82",2.71,1.68,8.83,"54,615,000,000.00",65.40,2.36
6,2007,France,"2,655,816,911,866.56",2.53,1.49,8.01,"63,890,000,000.00",65.48,2.41
7,2008,France,"2,926,802,941,585.86",0.38,2.81,7.39,"68,034,000,000.00",69.82,2.32
8,2009,France,"2,700,075,882,518.98",-2.82,0.09,9.12,"58,906,000,000.00",84.06,2.18
9,2010,France,"2,646,230,027,988.34",2.00,1.53,9.28,"56,178,000,000.00",86.28,2.12


In [7]:
#Tourist arrival info was unavialble for France since there was a change in data collection and measures

# SECTION 8: CLEAN INTERNATIONAL TOURIST-ARRIVAL DATA
# Load international tourist-arrival data from Our World in Data
url_UN = "https://ourworldindata.org/grapher/international-tourist-trips.csv?v=1&csvType=full&useColumnShortNames=false"


df_UN_Tourist_Arrivals = pd.read_csv("international-tourist-trips.csv")

# Keep Germany, France, and the 2001-2011 study period
tourist_arrivals_fixed = df_UN_Tourist_Arrivals[
    (df_UN_Tourist_Arrivals["Entity"].isin(["Germany", "France"])) &
    (df_UN_Tourist_Arrivals["Year"].between(2001, 2011))
].copy()

# Select only the fields required for the final dataset
tourist_arrivals_fixed = tourist_arrivals_fixed[[
    "Entity",
    "Year",
    "Arrivals of tourists from abroad"
]]

# Standardize column names so they match the main dataset
tourist_arrivals_fixed.columns = [
    "country",
    "year",
    "tourist_arrivals"
]

# Organize the rows and create a clean sequential index
tourist_arrivals_fixed = tourist_arrivals_fixed.sort_values(
    ["country", "year"]
).reset_index(drop=True)

tourist_arrivals_fixed

,country,year,tourist_arrivals
0,France,2001,"75,202,000.00"
1,France,2002,"77,012,000.00"
2,France,2003,"75,048,000.00"
3,France,2004,"74,433,000.00"
4,France,2005,"74,988,000.00"
5,France,2006,"77,916,000.00"
6,France,2007,"80,853,000.00"
7,France,2008,"79,218,000.00"
8,France,2009,"76,764,000.00"
9,France,2010,"76,647,000.00"


In [8]:
# SECTION 9: CREATE THE FINAL WORLD CUP DATASET
# Add tourist arrivals to the macroeconomic and tourism-revenue data
worldcup_df = germany_france_df.merge(
    tourist_arrivals_fixed,
    on=["country", "year"],
    how="left"  # Preserve every row from the main country dataset
)

worldcup_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_revenue_pct_gdp,tourist_arrivals
0,2001,France,"1,370,376,677,298.86",1.90,1.63,8.61,"37,826,000,000.00",59.32,2.76,"75,202,000.00"
1,2002,France,"1,492,427,756,382.43",1.07,1.92,8.70,"41,113,000,000.00",61.26,2.75,"77,012,000.00"
2,2003,France,"1,835,095,983,049.09",0.97,2.10,8.31,"46,576,000,000.00",65.36,2.54,"75,048,000.00"
3,2004,France,"2,109,792,297,237.03",2.87,2.14,8.91,"53,068,000,000.00",66.88,2.52,"74,433,000.00"
4,2005,France,"2,192,146,403,028.17",1.89,1.75,8.88,"52,126,000,000.00",68.19,2.38,"74,988,000.00"
5,2006,France,"2,317,861,544,690.82",2.71,1.68,8.83,"54,615,000,000.00",65.40,2.36,"77,916,000.00"
6,2007,France,"2,655,816,911,866.56",2.53,1.49,8.01,"63,890,000,000.00",65.48,2.41,"80,853,000.00"
7,2008,France,"2,926,802,941,585.86",0.38,2.81,7.39,"68,034,000,000.00",69.82,2.32,"79,218,000.00"
8,2009,France,"2,700,075,882,518.98",-2.82,0.09,9.12,"58,906,000,000.00",84.06,2.18,"76,764,000.00"
9,2010,France,"2,646,230,027,988.34",2.00,1.53,9.28,"56,178,000,000.00",86.28,2.12,"76,647,000.00"


In [9]:
# Review the complete dataset after adding the period classification
worldcup_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_revenue_pct_gdp,tourist_arrivals
0,2001,France,"1,370,376,677,298.86",1.90,1.63,8.61,"37,826,000,000.00",59.32,2.76,"75,202,000.00"
1,2002,France,"1,492,427,756,382.43",1.07,1.92,8.70,"41,113,000,000.00",61.26,2.75,"77,012,000.00"
2,2003,France,"1,835,095,983,049.09",0.97,2.10,8.31,"46,576,000,000.00",65.36,2.54,"75,048,000.00"
3,2004,France,"2,109,792,297,237.03",2.87,2.14,8.91,"53,068,000,000.00",66.88,2.52,"74,433,000.00"
4,2005,France,"2,192,146,403,028.17",1.89,1.75,8.88,"52,126,000,000.00",68.19,2.38,"74,988,000.00"
5,2006,France,"2,317,861,544,690.82",2.71,1.68,8.83,"54,615,000,000.00",65.40,2.36,"77,916,000.00"
6,2007,France,"2,655,816,911,866.56",2.53,1.49,8.01,"63,890,000,000.00",65.48,2.41,"80,853,000.00"
7,2008,France,"2,926,802,941,585.86",0.38,2.81,7.39,"68,034,000,000.00",69.82,2.32,"79,218,000.00"
8,2009,France,"2,700,075,882,518.98",-2.82,0.09,9.12,"58,906,000,000.00",84.06,2.18,"76,764,000.00"
9,2010,France,"2,646,230,027,988.34",2.00,1.53,9.28,"56,178,000,000.00",86.28,2.12,"76,647,000.00"


In [10]:
# SECTION 10: CREATE BEFORE/AFTER PERIODS FROM THE YEAR COLUMN

# Years before the 2006 World Cup are labeled "Before"
# Years from 2006 onward are labeled "After"
worldcup_df["period"] = np.where(
    worldcup_df["year"] < 2006,
    "Before",
    "After"
)

worldcup_df

,year,country,gdp_current_usd,gdp_growth_pct,inflation_pct,unemployment_pct,tourism_revenue_current_usd,government_debt_pct_gdp,tourism_revenue_pct_gdp,tourist_arrivals,period
0,2001,France,"1,370,376,677,298.86",1.90,1.63,8.61,"37,826,000,000.00",59.32,2.76,"75,202,000.00",Before
1,2002,France,"1,492,427,756,382.43",1.07,1.92,8.70,"41,113,000,000.00",61.26,2.75,"77,012,000.00",Before
2,2003,France,"1,835,095,983,049.09",0.97,2.10,8.31,"46,576,000,000.00",65.36,2.54,"75,048,000.00",Before
3,2004,France,"2,109,792,297,237.03",2.87,2.14,8.91,"53,068,000,000.00",66.88,2.52,"74,433,000.00",Before
4,2005,France,"2,192,146,403,028.17",1.89,1.75,8.88,"52,126,000,000.00",68.19,2.38,"74,988,000.00",Before
5,2006,France,"2,317,861,544,690.82",2.71,1.68,8.83,"54,615,000,000.00",65.40,2.36,"77,916,000.00",After
6,2007,France,"2,655,816,911,866.56",2.53,1.49,8.01,"63,890,000,000.00",65.48,2.41,"80,853,000.00",After
7,2008,France,"2,926,802,941,585.86",0.38,2.81,7.39,"68,034,000,000.00",69.82,2.32,"79,218,000.00",After
8,2009,France,"2,700,075,882,518.98",-2.82,0.09,9.12,"58,906,000,000.00",84.06,2.18,"76,764,000.00",After
9,2010,France,"2,646,230,027,988.34",2.00,1.53,9.28,"56,178,000,000.00",86.28,2.12,"76,647,000.00",After


In [11]:
# SECTION 11: CALCULATE AVERAGE INDICATORS BY COUNTRY AND PERIOD
# Select the indicators used to compare the Before and After periods
metrics = [
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_pct",
    "government_debt_pct_gdp",
    "tourism_revenue_current_usd",
    "tourist_arrivals"
]

# Calculate the mean of each indicator for every country-period combination
avg_table = worldcup_df.pivot_table(
    index="country",
    columns="period",
    values=metrics,
    aggfunc="mean"
)

# Round the displayed results while keeping the underlying values unchanged
avg_table.round(2)

gdp_growth_pct        government_debt_pct_gdp        inflation_pct  \
period           After Before                   After Before         After   
country                                                                      
France            1.21   1.74                   76.63  64.20          1.62   
Germany           1.67   0.59                   71.17  62.66          1.67   

               tourism_revenue_current_usd                   tourist_arrivals  \
period  Before                       After            Before            After   
country                                                                         
France    1.91           61,292,666,666.67 46,141,800,000.00    78,649,500.00   
Germany   1.53           49,727,000,000.00 31,592,600,000.00    25,390,500.00   

                      unemployment_pct         
period         Before            After Before  
country                                        
France  75,336,600.00             8.64   8.68  
Germany 19,173,200.00             7.90   9.59

In [12]:
# SECTION 12: MEASURE THE CHANGE FROM BEFORE TO AFTER
# Extract the country averages for each period from the pivot table

#Cross sections (xs) works this way:
    #   dataframe.xs(
    #       label_to_find,
    #       level=where_to_find_it, --> in this case
    #        axis=rows_or_columns
    #   )

before = avg_table.xs(
    "Before",
    level=1,
    axis=1
)

after = avg_table.xs(
    "After",
    level=1,
    axis=1
)

# Positive values indicate an increase; negative values indicate a decrease
change_table = after - before

change_table.round(2)

,gdp_growth_pct,government_debt_pct_gdp,inflation_pct,tourism_revenue_current_usd,tourist_arrivals,unemployment_pct
country,,,,,,
France,-0.53,12.43,-0.29,"15,150,866,666.67","3,312,900.00",-0.04
Germany,1.08,8.51,0.14,"18,134,400,000.00","6,217,300.00",-1.69


In [13]:
# SECTION 13: ESTIMATE GERMANY'S RELATIVE WORLD CUP IMPACT
# Subtract France's change from Germany's change to create a comparison group
impact_table = (
    change_table.loc["Germany"]
    -
    change_table.loc["France"]
)

# Convert the resulting Series into a clearly labelled one-column table
impact_table = impact_table.to_frame(
    name="Germany_minus_France"
)

impact_table.round(2)

,Germany_minus_France
gdp_growth_pct,1.61
government_debt_pct_gdp,-3.92
inflation_pct,0.43
tourism_revenue_current_usd,"2,983,533,333.33"
tourist_arrivals,"2,904,400.00"
unemployment_pct,-1.65


In [14]:
#EXTRA CODE FOR GERMAN WORLD CUP STUDY 

# Germany 2006 World Cup - Estimated Total Investment ≈ €3.0 Billion (2006) ≈ $6.0 Billion (2025 USD)
#       “Germany invested €1.401 billion in stadium construction and renovation for the 2006 FIFA World Cup.”  
#       “Additional transport and infrastructure investments were estimated at roughly €1.6 billion according to the study.”


#Pivots to get
#       GDP 
#       Tourism Revenue
#       Tourist Arrivals
#       % GDP growth
#       % Government debt as part of GDP
#       % Tourism as part of GDP


In [15]:
# Create a GDP table with years as rows and countries as columns
# This makes it easy to compare Germany and France GDP year by year
gdp_table = worldcup_df.pivot_table(
    index="year",
    columns="country",
    values="gdp_current_usd",
    aggfunc="mean"
)

gdp_table

country,France,Germany
year,,
2001,"1,370,376,677,298.86","1,966,381,496,641.73"
2002,"1,492,427,756,382.43","2,102,350,798,305.89"
2003,"1,835,095,983,049.09","2,534,715,518,349.01"
2004,"2,109,792,297,237.03","2,852,317,768,061.78"
2005,"2,192,146,403,028.17","2,893,393,187,361.87"
2006,"2,317,861,544,690.82","3,046,308,753,670.58"
2007,"2,655,816,911,866.56","3,484,056,680,854.91"
2008,"2,926,802,941,585.86","3,808,197,720,125.00"
2009,"2,700,075,882,518.98","3,478,545,516,683.59"


In [16]:
# Create a tourism revenue table with years as rows and countries as columns
# This shows how much tourism revenue each country earned each year
tourism_revenue_table = worldcup_df.pivot_table(
    index="year",
    columns="country",
    values="tourism_revenue_current_usd",
    aggfunc="mean"
)

tourism_revenue_table

country,France,Germany
year,,
2001,"37,826,000,000.00","24,182,000,000.00"
2002,"41,113,000,000.00","26,717,000,000.00"
2003,"46,576,000,000.00","30,129,000,000.00"
2004,"53,068,000,000.00","36,417,000,000.00"
2005,"52,126,000,000.00","40,518,000,000.00"
2006,"54,615,000,000.00","45,560,000,000.00"
2007,"63,890,000,000.00","49,320,000,000.00"
2008,"68,034,000,000.00","53,402,000,000.00"
2009,"58,906,000,000.00","47,499,000,000.00"


In [17]:
# Create a tourist arrivals table with years as rows and countries as columns
# This compares the number of international tourist arrivals each year
tourist_arrivals_table = worldcup_df.pivot_table(
    index="year",
    columns="country",
    values="tourist_arrivals",
    aggfunc="mean"
)

tourist_arrivals_table

country,France,Germany
year,,
2001,"75,202,000.00","17,861,000.00"
2002,"77,012,000.00","17,969,000.00"
2003,"75,048,000.00","18,399,000.00"
2004,"74,433,000.00","20,137,000.00"
2005,"74,988,000.00","21,500,000.00"
2006,"77,916,000.00","23,569,000.00"
2007,"80,853,000.00","24,421,000.00"
2008,"79,218,000.00","24,884,000.00"
2009,"76,764,000.00","24,220,000.00"


In [18]:
# Create a GDP growth table with years as rows and countries as columns
# This compares each country's yearly GDP growth percentage
gdp_growth_table = worldcup_df.pivot_table(
    index="year",
    columns="country",
    values="gdp_growth_pct",
    aggfunc="mean"
)

gdp_growth_table

country,France,Germany
year,,
2001,1.90,1.64
2002,1.07,-0.23
2003,0.97,-0.53
2004,2.87,1.16
2005,1.89,0.89
2006,2.71,3.87
2007,2.53,2.89
2008,0.38,0.89
2009,-2.82,-5.54


In [19]:
# Create a government debt table with years as rows and countries as columns
# This compares government debt as a percentage of GDP for each country
gvt_debt= worldcup_df.pivot_table(
    index="year",
    columns= ["country"],
    values="government_debt_pct_gdp",
)

gvt_debt

country,France,Germany
year,,
2001,59.32,58.06
2002,61.26,59.82
2003,65.36,63.34
2004,66.88,64.96
2005,68.19,67.12
2006,65.40,66.40
2007,65.48,63.69
2008,69.82,65.16
2009,84.06,72.34


In [20]:
# Create a tourism revenue percentage-of-GDP table by year and country
# This shows how important tourism revenue was relative to each country's GDP
tourism_pct_gdp = worldcup_df.pivot_table(
    index="year",
    columns= ["country"],
    values="tourism_revenue_pct_gdp",
)

tourism_pct_gdp

country,France,Germany
year,,
2001,2.76,1.23
2002,2.75,1.27
2003,2.54,1.19
2004,2.52,1.28
2005,2.38,1.40
2006,2.36,1.50
2007,2.41,1.42
2008,2.32,1.40
2009,2.18,1.37


In [21]:
# Function to find the percentage change between 2 years in a dataframe.

def calculate_percent_change(df, start_year, end_year, value_column):
    start_value = df.loc[df["year"] == start_year, value_column].iloc[0]
    end_value = df.loc[df["year"] == end_year, value_column].iloc[0]

    return ((end_value - start_value) / start_value) * 100

In [22]:
calculate_percent_change(germany_df,2001, 2005,"gdp_current_usd" )

np.float64(47.14302348264211)

In [23]:
#WebScraping the image of wikipedia

import requests
from bs4 import BeautifulSoup

url_german_worldcup = "https://en.wikipedia.org/wiki/2006_FIFA_World_Cup"

headers = {
    "User-Agent": "WorldCupBootcampProject/1.0 (your-email@example.com)"
}

response = requests.get(
    url_german_worldcup,
    headers=headers,
    timeout=30
)

print(response.status_code)
print(response.url)

response.raise_for_status()


#We parse it
soup = BeautifulSoup(response.text, "html.parser")

#find all images
images = soup.find_all("img")

print(len(images)) #these are all the images on the website

#We'll look for the image target which has the name 2006_FIFA_World_Cup.svg
target_image = None

for img in images:
    src = img.get("src")
    
    if src and "2006_FIFA_World_Cup.svg" in src:
        target_image = src
        break

print (target_image)

#Wikipedia's image urls need to be fixed, needs to start with http:

if target_image.startswith("//"):
    target_image = "https:" + target_image

print(target_image)

target_image



200
https://en.wikipedia.org/wiki/2006_FIFA_World_Cup
734
//upload.wikimedia.org/wikipedia/en/thumb/6/6b/2006_FIFA_World_Cup.svg/250px-2006_FIFA_World_Cup.svg.png
https://upload.wikimedia.org/wikipedia/en/thumb/6/6b/2006_FIFA_World_Cup.svg/250px-2006_FIFA_World_Cup.svg.png


'https://upload.wikimedia.org/wikipedia/en/thumb/6/6b/2006_FIFA_World_Cup.svg/250px-2006_FIFA_World_Cup.svg.png'

In [24]:
from IPython.display import Image, display
display(Image(url=target_image))
